In [7]:
import ee
import time

# Initialize the Earth Engine library
ee.Initialize()

In [8]:
# 1. LOAD DHS CLUSTERS
ASSET_ID = 'projects/integrated-hawk-485001-k3/assets/PH_DHS_GPS'
dhs_points = ee.FeatureCollection(ASSET_ID)

In [9]:
# 2. DEFINE ADAPTIVE BUFFER FUNCTION
def adaptive_buffer(feature):
    urban_rural_status = ee.String(feature.get('URBAN_RURA'))
    is_urban = urban_rural_status.compareTo('U').eq(0)
    # 2000m for Urban, 5000m for Rural
    radius = ee.Number(ee.Algorithms.If(is_urban, 2000, 5000))
    return feature.buffer(radius).bounds()

dhs_squares = dhs_points.map(adaptive_buffer)

In [10]:
# 3. DEFINE CLOUD MASKING
def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask).divide(10000)

In [11]:
# 4. DEFINE QUARTERS
quarters = {
    1: ('2022-01-01', '2022-03-31')
    #2: ('2022-04-01', '2022-06-30'),
    #3: ('2022-07-01', '2022-09-30'),
    #4: ('2022-10-01', '2022-12-31')
}

In [12]:
# 5. EXPORT LOOP
features_list = dhs_squares.getInfo()['features']
total_tasks = 0

print(f"Found {len(features_list)} clusters. Starting batch export queue...")

for feature in features_list:
    cluster_id = str(feature['properties']['DHSCLUST'])
    
    # Reconstruct the specific adaptive geometry
    roi_geometry = ee.Geometry.Polygon(feature['geometry']['coordinates'])
    
    for q_num, (start_date, end_date) in quarters.items():
        
        # Load the collection with your 80% cloud limit
        quarterly_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(roi_geometry)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80)))
            
        # Your specific composite unmasking logic
        best_layer = (quarterly_col
                      .map(mask_s2_clouds)
                      .select(['B4', 'B3', 'B2', 'B8', 'B11'])
                      .median())
        
        backup_layer = (quarterly_col
                        .select(['B4', 'B3', 'B2', 'B8', 'B11'])
                        .mosaic()
                        .divide(10000))
        
        final_img = best_layer.unmask(backup_layer).clip(roi_geometry)
        
        filename = f"dhs_{cluster_id}_2022_Q{q_num}"
        
        # Send the task to Google Drive
        export_task = ee.batch.Export.image.toDrive(
            image=final_img,
            description=filename,
            folder='Sentinel2_Training_Data',
            region=roi_geometry,
            scale=10,
            crs='EPSG:3857', # Your original projection
            fileFormat='GeoTIFF'
        )
        
        export_task.start()
        total_tasks += 1

print(f"Successfully queued {total_tasks} tasks in Earth Engine!")

Found 1247 clusters. Starting batch export queue...


EEException: Too many tasks already in the queue (3000, limit 3000).